# Avaliação automática de frases — gramática, estrutura, semântica e coerência

Dada **uma frase qualquer**, sem gabarito, sem referência e sem saber quem a escreveu (pessoa,
tradutor, OCR, modelo), quanto dá para dizer sobre a qualidade dela — e com que confiança?

A pergunta parece uma só, mas são cinco perguntas independentes, e cada uma exige um instrumento
diferente:

| dimensão | pergunta | nível |
|---|---|---|
| **ortografia / mecânica** | as palavras existem? pontuação e maiúsculas estão certas? | frase |
| **gramática** | concordância, regência, tempos verbais estão corretos? | frase |
| **estrutura** | tem verbo principal? é fragmento? o período é navegável? | frase |
| **semântica** | a frase *significa* alguma coisa? | frase |
| **coesão / coerência** | as partes se encadeiam? | **texto** |

Repare na última linha: **coerência não é propriedade de uma frase isolada.** "O carro é azul" não
é coerente nem incoerente — é só uma frase. Coerência é uma relação *entre* frases. Por isso este
notebook trabalha em dois níveis, e a Parte II só existe a partir da segunda frase.

## O eixo: dimensão × juiz

O que organiza o material não é "com o que a frase é comparada" — uma frase solta não tem com o
que ser comparada. É o cruzamento entre **o que se mede** e **quem mede**:

| juiz | custo | dá diagnóstico? | onde serve |
|---|---|---|---|
| léxico + regra | ~0 | sim, exato | ortografia, mecânica |
| parser morfossintático | baixo | sim, aponta o token | gramática, estrutura |
| modelo de linguagem | médio | não, só um número | aceitabilidade, semântica, coerência |
| LLM com rubrica | alto | sim, em linguagem natural | todas, com ressalvas |
| humano | altíssimo | sim | o padrão-ouro |

A tensão entre as duas colunas do meio é o fio condutor: **o parser diz exatamente o que está
errado, mas é cego ao significado; o modelo de linguagem percebe que algo está errado, mas não
sabe dizer o quê.** Nenhum dos dois sozinho é um avaliador.

## Método: pares mínimos

Cada dimensão traz um exemplo onde o instrumento é o certo (✅) e um onde ele falha de forma
característica (❌) — conhecer o modo de falha é metade de saber usar.

Para isso o notebook usa **pares mínimos**: uma frase-base correta e versões dela com **uma única
corrupção**, uma por dimensão. É a metodologia do CoLA e do BLiMP, e ela permite a pergunta que
realmente importa: *este detector prefere a frase certa à errada?* Se a única diferença entre duas
frases é a concordância, e o detector as ordena corretamente, ele está medindo concordância — e
não comprimento, vocabulário ou assunto.

> **Custo:** ortografia, gramática e estrutura são instantâneas (spaCy + léxico). Aceitabilidade,
> semântica, coerência e o juiz usam Qwen2.5-1.5B na CPU — a célula de carga leva ~1 minuto e o
> teste de embaralhamento, alguns minutos.

In [1]:
!pip install spacy pyspellchecker
!python -m spacy download pt_core_news_sm

import re, math, itertools
from collections import Counter

import spacy
from spellchecker import SpellChecker

nlp = spacy.load("pt_core_news_sm")
lexico = SpellChecker(language="pt")
TOTAL_TOKENS = lexico.word_frequency.total_words

print(f"spaCy pt_core_news_sm  |  léxico: {len(lexico.word_frequency.dictionary):,} palavras, "
      f"{TOTAL_TOKENS:,} tokens")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 10.1 MB/s  0:00:01m0:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
spaCy pt_core_news_sm  |  léxico: 416,782 palavras, 387,750,743 tokens


In [2]:
BASE = "Os pesquisadores analisaram os dados coletados durante o experimento."

frases = {
    "correta":       BASE,
    "ortografia":    "Os pesquisadores analizaram os dados coletados durante o experimento.",
    "conc. nominal": "Os pesquisadores analisaram os dados coletado durante o experimento.",
    "conc. verbal":  "Os pesquisadores analisou os dados coletados durante o experimento.",
    "fragmento":     "Os pesquisadores que analisaram os dados coletados durante o experimento.",
    "anomalia sem.": "Os pesquisadores beberam os dados coletados durante o experimento.",
    "embaralhada":   "Dados os analisaram pesquisadores os durante coletados experimento o.",
    "vazia":         "Ideias verdes incolores dormem furiosamente.",
}

defeito = {
    "correta": None, "ortografia": "ortografia", "conc. nominal": "gramática",
    "conc. verbal": "gramática", "fragmento": "estrutura", "anomalia sem.": "semântica",
    "embaralhada": "estrutura", "vazia": "semântica",
}

for nome, f in frases.items():
    print(f"  {nome:14} {f}")

print("\nDuas observações sobre o corpus:")
print("  'embaralhada' e 'vazia' NÃO são pares mínimos da base — são sondas extras.")
print("  'vazia' (Chomsky, 1957) é perfeitamente gramatical e não significa nada:")
print("  é a frase que separa um verificador de gramática de um verificador de sentido.")

  correta        Os pesquisadores analisaram os dados coletados durante o experimento.
  ortografia     Os pesquisadores analizaram os dados coletados durante o experimento.
  conc. nominal  Os pesquisadores analisaram os dados coletado durante o experimento.
  conc. verbal   Os pesquisadores analisou os dados coletados durante o experimento.
  fragmento      Os pesquisadores que analisaram os dados coletados durante o experimento.
  anomalia sem.  Os pesquisadores beberam os dados coletados durante o experimento.
  embaralhada    Dados os analisaram pesquisadores os durante coletados experimento o.
  vazia          Ideias verdes incolores dormem furiosamente.

Duas observações sobre o corpus:
  'embaralhada' e 'vazia' NÃO são pares mínimos da base — são sondas extras.
  'vazia' (Chomsky, 1957) é perfeitamente gramatical e não significa nada:
  é a frase que separa um verificador de gramática de um verificador de sentido.


---
# Parte I — Nível da frase

## Dimensão 1 — Ortografia e mecânica

O andar térreo: as palavras existem? A frase começa com maiúscula, termina com pontuação, não tem
espaço antes da vírgula nem palavra duplicada?

O juiz é um **léxico** (lista de palavras válidas) mais **regras determinísticas**. Um corretor de
verdade acrescenta *distância de edição* (Levenshtein) para sugerir a correção — "analizaram" está
a uma substituição de "analisaram" — e usa a **frequência** da palavra para escolher entre
candidatas.

Duas propriedades desta camada, e as duas importam para o resto do notebook:

1. É a única dimensão em que o veredito é **exato**: ou a palavra está no dicionário, ou não está.
2. É **pré-requisito** das camadas estatísticas. Uma palavra fora do vocabulário desregula qualquer
   medida baseada em frequência — a Dimensão 4 vai mostrar isso acontecendo de forma concreta.

In [3]:
def tokenizar(texto):
    return re.findall(r"[a-zà-ÿ]+", texto.lower())


def erros_ortograficos(texto):
    fora = [p for p in tokenizar(texto) if p not in lexico]
    return {p: lexico.correction(p) for p in fora}


def erros_mecanicos(texto):
    problemas = []
    t = texto.strip()
    if t and not t[0].isupper():
        problemas.append("não começa com maiúscula")
    if t and t[-1] not in ".!?":
        problemas.append("sem pontuação final")
    if re.search(r"\s+[,.;:!?]", t):
        problemas.append("espaço antes de pontuação")
    palavras = tokenizar(t)
    dup = [a for a, b in zip(palavras, palavras[1:]) if a == b]
    if dup:
        problemas.append(f"palavra duplicada: {dup}")
    return problemas


print(f"{'frase':14} {'desconhecidas (→ sugestão)':42} mecânica")
for nome, f in frases.items():
    orto = erros_ortograficos(f)
    txt = ", ".join(f"{p} → {c}" for p, c in orto.items()) or "—"
    print(f"{nome:14} {txt:42} {erros_mecanicos(f) or '—'}")

print("\n--- palavras que existem, no lugar errado ---")
armadilhas = [
    "Ele estudou mas não passou na prova.",      # correto
    "Ele estudou mais não passou na prova.",     # 'mais' no lugar de 'mas'
    "Há dois anos que não o vejo.",              # correto
    "A dois anos que não o vejo.",               # 'a' no lugar de 'há'
]
for f in armadilhas:
    print(f"  {str(erros_ortograficos(f) or 'nenhuma desconhecida'):28} {f}")

frase          desconhecidas (→ sugestão)                 mecânica
correta        —                                          —
ortografia     analizaram → analisaram                    —
conc. nominal  —                                          —
conc. verbal   —                                          —
fragmento      —                                          —
anomalia sem.  —                                          —
embaralhada    —                                          —
vazia          —                                          —

--- palavras que existem, no lugar errado ---
  nenhuma desconhecida         Ele estudou mas não passou na prova.
  nenhuma desconhecida         Ele estudou mais não passou na prova.
  nenhuma desconhecida         Há dois anos que não o vejo.
  nenhuma desconhecida         A dois anos que não o vejo.


### Lendo os resultados

✅ **Onde funciona:** "analizaram" é pego de imediato e a sugestão por frequência acerta a
correção. Nenhum modelo, nenhuma GPU, resposta em microssegundos, e o diagnóstico é *exato* — não
"esta frase parece ruim", mas "esta palavra, aqui, não existe". Nenhuma camada posterior deste
notebook consegue ser tão precisa.

❌ **Onde falha — o erro escrito com palavras que existem.** As quatro armadilhas passam **todas**
sem uma única palavra desconhecida. "Ele estudou **mais** não passou" e "**A** dois anos que não o
vejo" são erros que qualquer revisor marca, e o dicionário não vê nada: `mais` e `a` estão no
léxico. É a distinção entre **erro de grafia** (a palavra não existe) e **erro de escolha** (a
palavra existe, mas não é essa) — e a segunda exige contexto, ou seja, exige as próximas camadas.

O outro limite é o inverso: nome próprio, termo técnico e estrangeirismo são reportados como
desconhecidos sem serem erros. Todo corretor precisa de uma lista de exceções, e essa lista nunca
termina.

**O que esta camada estabelece:** a ortografia é a única dimensão com veredito binário confiável —
e vem **primeiro no pipeline**, porque tudo o que é estatístico depende de palavras conhecidas.

## Dimensão 2 — Gramática (concordância)

Aqui a análise deixa de ser sobre palavras isoladas e passa a ser sobre **relações entre palavras**.
Concordância é, literalmente, um teste de igualdade entre traços morfológicos de dois tokens
ligados na árvore sintática:

- **concordância nominal:** determinante e adjetivo casam em gênero e número com o núcleo do sintagma
  — *os dados coletados*, não *os dados coletado*;
- **concordância verbal:** o verbo casa em número e pessoa com o sujeito — *os pesquisadores
  analisaram*, não *os pesquisadores analisou*.

O juiz é um **parser morfossintático**. O spaCy devolve, para cada token: a classe (`pos_`), os
traços morfológicos (`morph` — `Number`, `Gender`, `Person`), a função sintática (`dep_`) e o
**head**, o token do qual ele depende. Com isso, a verificação vira um percurso da árvore: para
cada `det`/`amod`/`acl`, compare com seu head; para cada `nsubj`, compare com o verbo.

É a abordagem por trás de ferramentas como o **LanguageTool**, que empacota milhares de regras
escritas à mão sobre uma análise assim.

In [4]:
def numero(t): return t.morph.get("Number")
def genero(t): return t.morph.get("Gender")


def erros_concordancia(doc):
    erros = []
    for t in doc:
        if t.dep_ in ("det", "amod", "acl") and t.head.pos_ in ("NOUN", "PROPN"):
            if numero(t) and numero(t.head) and numero(t) != numero(t.head):
                erros.append(f"nominal/número: '{t.text}'={numero(t)[0]} × '{t.head.text}'={numero(t.head)[0]}")
            if genero(t) and genero(t.head) and genero(t) != genero(t.head):
                erros.append(f"nominal/gênero: '{t.text}'={genero(t)[0]} × '{t.head.text}'={genero(t.head)[0]}")
        if t.dep_ in ("nsubj", "nsubj:pass") and t.head.pos_ in ("VERB", "AUX"):
            if numero(t) and numero(t.head) and numero(t) != numero(t.head):
                erros.append(f"verbal: sujeito '{t.text}'={numero(t)[0]} × verbo '{t.head.text}'={numero(t.head)[0]}")
    return erros


for nome in ["correta", "conc. verbal"]:
    print(f"--- {nome}: {frases[nome]}")
    for t in nlp(frases[nome]):
        if t.pos_ != "PUNCT":
            print(f"    {t.text:14} {t.pos_:6} {t.dep_:10} → {t.head.text:14} {t.morph}")
    print()

print(f"{'frase':14} erros de concordância")
for nome, f in frases.items():
    e = erros_concordancia(nlp(f))
    print(f"{nome:14} {'; '.join(e) if e else '—'}")

print("\n--- quando o próprio parser absorve o erro ---")
for f in ["Os menino analisaram os dados.",
          "Os menino que estudaram muito passou na prova."]:
    doc = nlp(f)
    traços = {t.text: t.morph.get("Number") for t in doc if t.dep_ == "det" or t.pos_ == "NOUN"}
    print(f"  {f}")
    print(f"     traços lidos: {traços}")
    print(f"     acusou:       {erros_concordancia(doc) or '— NADA'}")

--- correta: Os pesquisadores analisaram os dados coletados durante o experimento.
    Os             DET    det        → pesquisadores  Definite=Def|Gender=Masc|Number=Plur|PronType=Art
    pesquisadores  NOUN   nsubj      → analisaram     Gender=Masc|Number=Plur
    analisaram     VERB   ROOT       → analisaram     Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbForm=Fin
    os             DET    det        → dados          Definite=Def|Gender=Masc|Number=Plur|PronType=Art
    dados          NOUN   obj        → analisaram     Gender=Masc|Number=Plur
    coletados      VERB   acl        → dados          Gender=Masc|Number=Plur|VerbForm=Part
    durante        ADP    case       → experimento    
    o              DET    det        → experimento    Definite=Def|Gender=Masc|Number=Sing|PronType=Art
    experimento    NOUN   obl        → coletados      Gender=Masc|Number=Sing

--- conc. verbal: Os pesquisadores analisou os dados coletados durante o experimento.
    Os             DET    det

### Lendo os resultados

✅ **Onde funciona — e funciona muito bem.** As duas corrupções de concordância são pegas com
**localização exata do token e do traço violado**:

- `conc. nominal` → `'coletado'=Sing × 'dados'=Plur`
- `conc. verbal` → `sujeito 'pesquisadores'=Plur × verbo 'analisou'=Sing`

Isso é qualitativamente superior a qualquer nota. O detector não diz "0.7"; ele diz *qual palavra*,
*qual traço* e *contra o quê*. Para uma ferramenta de revisão, é a diferença entre um aviso
acionável e um número inútil. E custa milissegundos.

❌ **Onde falha — três modos, todos visíveis acima:**

1. **Cegueira semântica total.** `anomalia sem.` — "Os pesquisadores **beberam** os dados" — passa
   sem um único erro. A árvore é impecável, toda concordância bate. *Beber dados* é sintaticamente
   idêntico a *analisar dados*: sujeito plural, verbo plural, objeto direto. O parser valida a
   forma e não tem opinião sobre o sentido. Esta linha é a razão de existirem as Dimensões 4 e 5.

2. **Falso positivo.** `vazia` — "Ideias verdes incolores dormem furiosamente" — é acusada de erro
   de gênero em `'incolores' × 'verdes'`. A frase está **correta** (*incolor* e *verde* são
   invariáveis em gênero); o parser é que se atrapalhou com uma sequência improvável e etiquetou
   errado. Um detector construído sobre um modelo estatístico herda os erros dele, e num texto
   incomum — poesia, jargão, língua falada — os falsos positivos se multiplicam.

3. **O parser às vezes absorve o erro — e passa a acusar o token errado.** O último bloco mostra
   o **mesmo erro** ("Os menino") em dois contextos. Em *"Os menino analisaram os dados"*, o spaCy
   lê `menino` como `Sing` e o detector acerta em cheio: aponta a discordância com "Os" **e** com o
   verbo. Em *"Os menino que estudaram muito passou na prova"*, ele lê o **mesmo** `menino` como
   `Plur` — a discordância com "Os" **some do relatório**, e o que resta é uma acusação contra
   `passou`. Ou seja: o detector manda corrigir o **verbo** quando o erro está no **substantivo**.
   O motivo é que `morph` não é leitura da superfície, é **predição em contexto** — cercado de
   plurais, o modelo infere o que *deveria* estar escrito em vez de registrar o que está. Para uma
   ferramenta de revisão, diagnóstico deslocado é pior que silêncio: manda o autor mexer onde não
   deve.

**O que esta camada estabelece:** contra erro de forma, o parser é o melhor instrumento do notebook
— preciso, barato e explicativo. Contra erro de sentido, ele é inerte.

## Dimensão 3 — Estrutura

Estrutura é o esqueleto: a frase se sustenta como frase? As perguntas concretas:

- **Tem predicado?** Uma sentença precisa de um verbo finito na **raiz** da árvore. E o teste certo
  não é "existe algum verbo no texto" — é "a **raiz** é um verbo". A diferença é exatamente o que
  separa *"Os pesquisadores analisaram os dados"* de *"Os pesquisadores **que** analisaram os
  dados"*: ambas contêm o verbo `analisaram`, mas na segunda ele foi rebaixado a uma oração
  relativa e a raiz virou o substantivo `pesquisadores`. Sobra um sintagma nominal gigante, não
  uma frase. Um único token — `que` — converte oração em fragmento.
- **Quão emaranhada é?** A **profundidade da árvore** mede encaixamento: cada nível é uma
  subordinação a mais que o leitor precisa manter na memória.
- **Quão densa é?** Fórmulas de legibilidade combinam comprimento da frase e tamanho das palavras.
  Para o português, a adaptação do índice Flesch (Martins et al., 1996, base das ferramentas do
  NILC/USP) é:

$$\text{ILF} = 248{,}835 - 1{,}015 \times \frac{\text{palavras}}{\text{frases}} - 84{,}6 \times \frac{\text{sílabas}}{\text{palavras}}$$

  A constante muda em relação ao Flesch original (206,835) porque a palavra portuguesa é mais longa
  que a inglesa. Vale a ressalva: o índice foi calibrado para **textos**, não para frases isoladas —
  aplicado a uma frase só, o primeiro termo vira apenas o comprimento dela.

In [5]:
VOGAIS = "aeiouáéíóúâêôàãõ"


def contar_silabas(palavra):
    n, anterior_vogal = 0, False
    for c in palavra.lower():
        eh_vogal = c in VOGAIS
        if eh_vogal and not anterior_vogal:
            n += 1
        anterior_vogal = eh_vogal
    return max(1, n)


def flesch_pt(texto):
    sentencas = [s for s in re.split(r"[.!?]+", texto) if s.strip()]
    palavras = tokenizar(texto)
    if not sentencas or not palavras:
        return 0.0
    return (248.835
            - 1.015 * (len(palavras) / len(sentencas))
            - 84.6 * (sum(contar_silabas(p) for p in palavras) / len(palavras)))


def analisar_estrutura(doc):
    raiz = [t for t in doc if t.dep_ == "ROOT"][0]
    tem_predicado = raiz.pos_ in ("VERB", "AUX") or any(f.dep_ == "cop" for f in raiz.children)
    profundidade = max(len(list(t.ancestors)) for t in doc)
    return raiz, tem_predicado, profundidade


print(f"{'frase':14} {'raiz':16} {'predicado?':11} {'prof.':>5} {'Flesch-pt':>10}")
for nome, f in frases.items():
    doc = nlp(f)
    raiz, pred, prof = analisar_estrutura(doc)
    algum_verbo = any(t.pos_ in ("VERB", "AUX") for t in doc)
    marca = "FRAGMENTO" if not pred else "ok"
    print(f"{nome:14} {raiz.text + '/' + raiz.pos_:16} {marca:11} {prof:5} {flesch_pt(f):10.1f}")

print("\nnota: 'fragmento' CONTÉM um verbo ('analisaram'), mas ele não é a raiz —")
print("por isso o teste correto é sobre a RAIZ, não sobre a presença de verbo.")

print("\n--- o que a legibilidade realmente mede ---")
textos = {
    "curta e clara":     "O gato dormiu no sofá.",
    "técnica e clara":   "Os pesquisadores analisaram os dados coletados durante o experimento.",
    "longa e confusa":   ("A operacionalização metodológica da investigação pressupõe a "
                          "consideração sistemática das interdependências epistemológicas "
                          "subjacentes à conceituação previamente estabelecida."),
}
for nome, t in textos.items():
    print(f"  {nome:18} Flesch-pt = {flesch_pt(t):7.1f}")

frase          raiz             predicado?  prof.  Flesch-pt
correta        analisaram/VERB  ok              4      -14.1
ortografia     analizaram/VERB  ok              4      -14.1
conc. nominal  analisaram/VERB  ok              4      -14.1
conc. verbal   analisou/VERB    ok              4       -4.7
fragmento      pesquisadores/NOUN FRAGMENTO       5        1.8
anomalia sem.  beberam/VERB     ok              4        4.7
embaralhada    analisaram/VERB  ok              2      -14.1
vazia          dormem/VERB      ok              3      -10.0

nota: 'fragmento' CONTÉM um verbo ('analisaram'), mas ele não é a raiz —
por isso o teste correto é sobre a RAIZ, não sobre a presença de verbo.

--- o que a legibilidade realmente mede ---
  curta e clara      Flesch-pt =   108.4
  técnica e clara    Flesch-pt =   -14.1
  longa e confusa    Flesch-pt =  -106.8


### Lendo os resultados

✅ **Onde funciona:** o fragmento é identificado sem ambiguidade — a raiz de
*"Os pesquisadores que analisaram..."* é o substantivo `pesquisadores`, não um verbo. E o detalhe
impresso abaixo da tabela é a lição da célula: a frase **contém** um verbo. Um teste ingênuo
("tem verbo?") aprova o fragmento; o teste sobre a **raiz** o reprova. A estrutura da árvore
carrega informação que a lista de palavras não carrega.

A profundidade também se comporta: o fragmento fica mais fundo que a base, porque o verbo desceu
um nível ao virar oração relativa.

❌ **Onde falha:** compare as duas primeiras linhas do último bloco. *"O gato dormiu no sofá"*
marca **108** e a frase técnica **correta e perfeitamente clara** marca **−14**. As duas estão
fora da escala teórica de 0–100, em direções opostas — o que já diz que o índice não deve ser
lido como nota. A fórmula não leu nada: ela contou sílabas. Palavra longa é penalizada, mesmo
quando é a palavra exata e o leitor é da área. Legibilidade mede **densidade lexical**, não
clareza — e é facilmente enganada nas duas direções: trocar termos precisos por perífrases vagas
*melhora* o índice enquanto piora o texto.

Repare também que `correta`, `ortografia` e `conc. nominal` recebem **exatamente** o mesmo −14,1:
a fórmula conta sílabas e palavras, e as três têm as mesmas. Erro de grafia e erro de concordância
são invisíveis para ela por construção.

Segunda falha, e mais séria: `embaralhada` — *"Dados os analisaram pesquisadores os durante
coletados experimento o."* — passa por **todos** os testes estruturais. Raiz verbal: ok.
Predicado: ok. E a profundidade é **2**, a *menor* de todo o corpus: pela métrica, é a frase mais
simples e mais bem estruturada que temos. É lixo.

A causa é fundamental: **o parser impõe *alguma* árvore a qualquer sequência de palavras.** Ele
nunca responde "isto não é analisável" — sempre devolve uma análise, e quanto mais incoerente a
entrada, mais rasa e mais inocente a árvore resultante. Métrica estrutural derivada de parse herda
essa cegueira. Detectar que a análise é absurda exige outro juiz: o da próxima dimensão, que dá a
`embaralhada` um SLOR **negativo**.

## Dimensão 4 — Aceitabilidade estatística (SLOR)

Mesma dimensão de antes (a frase é bem formada?), **juiz completamente diferente**. Em vez de
regras sobre uma árvore, um modelo de linguagem: quão provável é esta sequência de palavras?

A ideia tem um nome na literatura — *acceptability judgment* — e uma armadilha grande. A medida
óbvia, a **perplexidade**, não serve, por um motivo simples: ela confunde **raro** com **errado**.
Uma frase correta cheia de termos técnicos é improvável porque as palavras são improváveis, não
porque a gramática esteja quebrada.

A correção é o **SLOR** (*Syntactic Log-Odds Ratio*, Pauls & Klein, 2012; popularizado para
fluência por Lau, Clark & Lappin):

$$\text{SLOR}(s) = \frac{\ln P_{\text{LM}}(s) - \ln P_{\text{unigrama}}(s)}{|s|}$$

Lê-se: da probabilidade que o modelo dá à frase, **desconte** a probabilidade que ela já teria só
pela frequência isolada das palavras, e normalize pelo comprimento. O que sobra é o que a
*estrutura* acrescenta — palavra rara deixa de ser punida, porque ela é rara nos dois termos e o
efeito se cancela. As frequências vêm da mesma tabela do léxico da Dimensão 1.

**A regra de uso que decide tudo:** SLOR é confiável **entre pares mínimos** — mesmo conteúdo, uma
perturbação. Comparar frases de assuntos diferentes por SLOR não significa grande coisa.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForMaskedLM

MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32)
modelo.eval()

# BERTimbau: modelo mascarado, usado na etapa semantica da pipeline (verbo mascarado)
BERTIMBAU = "neuralmind/bert-base-portuguese-cased"
tok_b = AutoTokenizer.from_pretrained(BERTIMBAU)
mod_b = AutoModelForMaskedLM.from_pretrained(BERTIMBAU); mod_b.eval()
print("modelos carregados (Qwen causal + BERTimbau mascarado)")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
def logprobs_tokens(texto):
    enc = tokenizer(texto, return_tensors="pt")
    ids = enc["input_ids"][0]
    with torch.no_grad():
        lp = torch.log_softmax(modelo(**enc).logits, dim=-1)
    return [(tokenizer.decode(ids[i]), float(lp[0, i - 1, ids[i]])) for i in range(1, len(ids))]


def perplexidade(texto):
    vs = [v for _, v in logprobs_tokens(texto)]
    return math.exp(-sum(vs) / len(vs))


def logprob_unigrama(texto):
    total = 0.0
    for p in tokenizar(texto):
        c = lexico.word_frequency[p] or 1
        total += math.log(c / TOTAL_TOKENS)
    return total


def slor(texto):
    lp = sum(v for _, v in logprobs_tokens(texto))
    return (lp - logprob_unigrama(texto)) / len(tokenizar(texto))


print(f"{'frase':14} {'PPL':>8} {'SLOR':>8}")
for nome, f in frases.items():
    print(f"{nome:14} {perplexidade(f):8.1f} {slor(f):8.2f}")

print("\n--- duas frases CORRETAS, vocabulários opostos ---")
par = {
    "banal (palavras comuns)": "Ele foi para casa e depois foi dormir.",
    "correta (palavras raras)": "O sismólogo catalogou anomalias geomagnéticas ininterruptamente.",
}
for nome, f in par.items():
    print(f"  {nome:26} PPL = {perplexidade(f):7.1f}   SLOR = {slor(f):5.2f}")

frase               PPL     SLOR


correta             8.8     5.82


ortografia         12.6     5.83


conc. nominal      14.8     4.91


conc. verbal       13.6     4.95


...

anomalia sem.      26.4     3.80


embaralhada       317.1    -0.54


vazia             312.3    -0.90

--- duas frases CORRETAS, vocabulários opostos ---


  banal (palavras comuns)    PPL =    61.1   SLOR =  1.28


  correta (palavras raras)   PPL =   113.9   SLOR =  2.01


### Lendo os resultados

✅ **Onde funciona — a correção de frequência é visível.** As duas frases do último bloco estão
**corretas**. A perplexidade coloca a técnica quase **duas vezes pior** que a banal (≈114 contra
≈61) — punição que é puro efeito de vocabulário, não de gramática. O SLOR não reproduz essa
punição: descontado o termo unigrama, a frase rara deixa de ser penalizada por ser rara. É
exatamente o que a métrica foi desenhada para fazer, e é a razão de não se usar perplexidade crua
para julgar frase.

E na tabela principal a **separação grossa** funciona: tudo que ainda é uma frase de português
fica na faixa ~4,9–5,8, a anomalia semântica cai para ~3,8, e as duas frases realmente quebradas
despencam para valores **negativos** — `vazia` (−0,90) e `embaralhada` (−0,54). Negativo significa
que a estrutura *tira* probabilidade em vez de acrescentar: o modelo teria previsto melhor
tratando as palavras como independentes. Para triagem — "isto é texto ou é lixo?" — o SLOR
resolve.

❌ **Onde falha — três problemas, e o primeiro é uma armadilha de implementação:**

1. **A frase com erro de ortografia pontua ligeiramente ACIMA da correta** (≈5,83 contra ≈5,82).
   Não é ruído: é o tratamento de palavra desconhecida. "analizaram" não está no léxico, cai na
   contagem-piso 1, e seu log-unigrama fica altíssimo em módulo — o que **infla o numerador do
   SLOR**. A fórmula recompensa o erro de grafia. É a demonstração concreta da promessa feita na
   Dimensão 1: **ortografia é pré-requisito, não irmã**. Rode o corretor primeiro; só aplique SLOR
   a texto dentro do vocabulário.

2. **O número não diz o que está errado.** `conc. nominal` (4,91), `conc. verbal` (4,95) e
   `fragmento` (4,93) ficam praticamente empatados. Três defeitos de naturezas distintas —
   morfologia, morfologia e sintaxe — colapsam no mesmo valor. O SLOR percebe que algo está pior;
   não tem como dizer o quê, nem onde. É o espelho exato do parser: lá havia diagnóstico sem
   percepção de sentido; aqui há percepção sem diagnóstico.

3. **A escala não é absoluta.** Não existe "SLOR > 4 = frase boa". O valor depende do modelo, da
   tabela de frequências e do domínio. Só a **comparação dentro de um par mínimo** é interpretável
   — e é assim que a Dimensão 8 vai medir os detectores.

## Dimensão 5 — Semântica

A frase está escrita corretamente. Ela **quer dizer** alguma coisa?

*"Os pesquisadores beberam os dados"* atravessou a ortografia, a concordância e a estrutura sem
um arranhão. O que está errado é uma **restrição de seleção**: o verbo *beber* exige um objeto
líquido, e *dados* não é líquido. A informação necessária para detectar isso não está na
morfologia nem na árvore — está no significado das palavras.

Três instrumentos, do mais barato ao mais caro:

- **Surpresa por token** (*surprisal*, $-\ln P(\text{token} \mid \text{contexto})$): o modelo já
  calcula isso. Onde a frase viola uma expectativa semântica, a surpresa **dispara naquele token**.
  A vantagem sobre o SLOR é decisiva: a surpresa é **localizada** — aponta a palavra.
- **Embeddings**: aproximar significados no espaço vetorial. Serve para similaridade e agrupamento,
  mas tem um ponto cego famoso, testado abaixo.
- **NLI / LLM**: perguntar a um modelo treinado se a frase é plausível. Caro, e é a Dimensão 7.

In [ ]:
def surpresa(texto, ignorar_primeiro=True):
    vs = [(t, -v) for t, v in logprobs_tokens(texto)]
    return vs[1:] if ignorar_primeiro else vs


for nome in ["correta", "anomalia sem."]:
    vs = surpresa(frases[nome])
    pico_tok, pico_val = max(vs, key=lambda x: x[1])
    print(f"--- {nome}")
    print("    " + "  ".join(f"{t.strip()}:{v:.1f}" for t, v in vs))
    print(f"    pico: '{pico_tok.strip()}' com {pico_val:.1f} nats\n")

tok0, val0 = max(surpresa(frases["correta"], ignorar_primeiro=False), key=lambda x: x[1])
print(f"sem descartar o 1º token, o pico da frase CORRETA é '{tok0.strip()}' com {val0:.1f} nats —")
print("mais que qualquer defeito real das outras frases, e não é defeito nenhum:")
print("sem contexto anterior, a primeira palavra é sempre imprevisível.")

import torch.nn.functional as F


def embedding(texto):
    enc = tokenizer(texto, return_tensors="pt")
    with torch.no_grad():
        h = modelo(**enc, output_hidden_states=True).hidden_states[-1][0]
    return h.mean(dim=0)


print("\n--- negação: uma palavra inverte o sentido ---")
a = "A área monitorada cresceu durante o período."
b = "A área monitorada não cresceu durante o período."
print(f"  cosseno entre afirmação e negação: "
      f"{float(F.cosine_similarity(embedding(a), embedding(b), dim=0)):.3f}   <- sentidos OPOSTOS")

--- correta
    quis:0.1  adores:0.0  anal:4.8  is:0.0  aram:0.0  os:2.5  dados:1.0  co:3.8  let:0.0  ados:0.0  durante:2.4  o:1.7  experiment:4.6  o:0.0  .:3.0
    pico: 'anal' com 4.8 nats



--- anomalia sem.
    quis:0.1  adores:0.0  be:14.0  ber:3.6  am:0.0  os:6.0  dados:3.4  co:4.7  let:0.0  ados:0.0  durante:2.7  o:1.5  experiment:2.7  o:0.0  .:2.7
    pico: 'be' com 14.0 nats



sem descartar o 1º token, o pico da frase CORRETA é 'pes' com 10.8 nats —
mais que qualquer defeito real das outras frases, e não é defeito nenhum:
sem contexto anterior, a primeira palavra é sempre imprevisível.

--- negação: uma palavra inverte o sentido ---


  cosseno entre afirmação e negação: 0.996   <- sentidos OPOSTOS


### Lendo os resultados

✅ **Onde funciona — e este é o melhor resultado do notebook.** A surpresa **localiza** a anomalia.
Na frase correta, o token do verbo (`anal`, de *analisaram*) custa ~4,8 nats; na frase anômala, o
token do verbo (`be`, de *beberam*) custa ~14,0 — quase o triplo, **exatamente no token culpado**.
Nenhuma das camadas anteriores tinha chegado perto: o parser não viu nada, o SLOR viu uma queda
sem endereço. A surpresa entrega o que faltava — *onde*.

Repare também no efeito de propagação: depois de "beberam", os tokens seguintes (`os`, `dados`)
também ficam mais caros que na frase correta. O modelo continua surpreso pelo resto da frase, e o
perfil de surpresa desenha o dano.

❌ **Onde falha — dois modos:**

1. **O primeiro token é sempre um pico falso.** Sem contexto anterior, `pes` (de *pesquisadores*)
   custa ~10,8 nats na frase **correta** — mais que qualquer defeito real no meio dela. Uma regra
   ingênua "o pico é o defeito" acusa a primeira palavra de toda frase perfeita. Por isso a função
   descarta o primeiro token, e é o tipo de artefato que faz um detector parecer funcionar em
   demonstração e falhar em produção.

2. **Surpresa alta ≠ erro.** A métrica mede o improvável, e o improvável inclui o *original*, o
   *técnico* e o *bem escrito*. Uma metáfora boa dispara o mesmo alarme que "beberam os dados". Não
   há como separar as duas coisas dentro desta dimensão — é preciso um juiz que entenda a diferença
   entre inovação e absurdo (Dimensão 7), ou aceitar a taxa de falsos positivos.

E o teste de negação repete um resultado clássico: "cresceu" e "não cresceu" dizem o **oposto** e
o cosseno mal se move. Sistemas que verificam afirmações por similaridade tropeçam exatamente aí.

**O que a pipeline usa.** A surpresa localiza, mas seu veredito depende de um limiar absoluto — e a
escala não é absoluta: o estudo mostra que verbos comuns como "cantou" e "ferveram" ficam abaixo do
corte mesmo sendo anômalos. Por isso a etapa semântica da pipeline (Parte IV) adota a técnica
**vencedora do estudo**, o **verbo mascarado**, que julga a plausibilidade do verbo-raiz por uma
**margem relativa** à posição, não por um número fixo. Ela tem o próprio ponto cego — só olha o
verbo —, que a matriz da Parte IV expõe.

---
# Parte II — Nível do texto

## Dimensão 6 — Coesão e coerência

As cinco dimensões anteriores cabem numa frase. Esta não cabe, e é por isso que ela precisa de
uma Parte própria. Duas noções que costumam ser confundidas:

- **Coesão** é de superfície: os elos visíveis entre frases — repetição de palavras, pronomes,
  conectivos ("por isso", "no entanto"), cadeias de referência.
- **Coerência** é de sentido: o texto se sustenta como um todo, as ideias progridem, cada frase se
  encaixa onde está.

Coesão é **necessária e insuficiente**. O experimento abaixo mostra as duas falhando em direções
opostas, com quatro parágrafos: um **coerente** (que usa sinônimos e conectivos), um **sem relação**
(quatro frases perfeitas sobre assuntos distintos), um **embaralhado** (o coerente fora de ordem) e
um **coeso mas vazio** (repete a mesma palavra em toda frase e não diz nada).

Dois medidores:

- **Sobreposição lexical entre frases vizinhas** — uma versão pobre do *entity grid*
  (Barzilay & Lapata), que modela coerência pelas transições de entidades entre frases.
- **Log-prob condicional** — quanto o modelo esperava cada frase **dadas as anteriores**. Uma
  continuação coerente é mais previsível que um salto de assunto.

E o protocolo padrão da área: o **teste de embaralhamento**. Um medidor de coerência que preste
tem de pontuar a ordem original acima das permutações. É objetivo, não precisa de anotação humana
e é usado exatamente assim na literatura.

Para o português, a referência em métricas desta família é o **Coh-Metrix-Port**, do NILC/USP.

In [ ]:
paragrafos = {
    "coerente": [
        "A equipe instalou sensores de temperatura na floresta em janeiro.",
        "Os aparelhos registraram medições a cada quinze minutos.",
        "Esses registros revelaram uma variação diária maior que a esperada.",
        "Por isso, o grupo decidiu revisar o modelo climático que usava.",
    ],
    "sem relação": [
        "A equipe instalou sensores de temperatura na floresta em janeiro.",
        "O preço do café subiu bastante no mercado internacional.",
        "Muitos gatos preferem dormir durante a tarde.",
        "A biblioteca municipal fecha às dezoito horas.",
    ],
    "coeso mas vazio": [
        "Os sensores são equipamentos importantes.",
        "Os sensores foram instalados pela equipe.",
        "Os sensores registram dados.",
        "Os sensores são muito utilizados.",
    ],
}
paragrafos["embaralhado"] = [paragrafos["coerente"][i] for i in (2, 0, 3, 1)]


def logprob_condicional(contexto, frase):
    n_ctx = tokenizer(contexto, return_tensors="pt")["input_ids"].shape[1] if contexto else 0
    enc = tokenizer(contexto + frase, return_tensors="pt")
    ids = enc["input_ids"][0]
    with torch.no_grad():
        lp = torch.log_softmax(modelo(**enc).logits, dim=-1)
    vs = [float(lp[0, i - 1, ids[i]]) for i in range(max(n_ctx, 1), len(ids))]
    return sum(vs) / len(vs)


def coerencia_lm(fs):
    return sum(logprob_condicional(" ".join(fs[:i]) + " ", fs[i]) for i in range(1, len(fs))) / (len(fs) - 1)


def substantivos(frase):
    return {t.lemma_.lower() for t in nlp(frase) if t.pos_ in ("NOUN", "PROPN")}


def coesao_lexical(fs):
    ov = [len(substantivos(fs[i]) & substantivos(fs[i + 1])) for i in range(len(fs) - 1)]
    return sum(1 for o in ov if o > 0) / len(ov), ov


print(f"{'parágrafo':18} {'coerência LM':>13} {'coesão lex.':>12}   sobreposições")
for nome, fs in paragrafos.items():
    c, ov = coesao_lexical(fs)
    print(f"{nome:18} {coerencia_lm(fs):13.3f} {c:12.2f}   {ov}")

parágrafo           coerência LM  coesão lex.   sobreposições


coerente                  -1.970         0.00   [0, 0, 0]


sem relação               -2.975         0.00   [0, 0, 0]


coeso mas vazio           -1.932         1.00   [1, 1, 1]


embaralhado               -2.326         0.00   [0, 0, 0]


In [ ]:
original = paragrafos["coerente"]
ranking = sorted(((coerencia_lm(list(p)), p) for p in itertools.permutations(original)),
                 key=lambda x: -x[0])
posicao = next(i for i, (s, p) in enumerate(ranking) if list(p) == original) + 1

print(f"ordem original: {posicao}º lugar entre {len(ranking)} permutações")
print(f"  melhor = {ranking[0][0]:.3f}   pior = {ranking[-1][0]:.3f}\n")
print("pior permutação encontrada:")
for f in ranking[-1][1]:
    print("   ", f)

ordem original: 1º lugar entre 24 permutações
  melhor = -1.970   pior = -2.509

pior permutação encontrada:
    Por isso, o grupo decidiu revisar o modelo climático que usava.
    Esses registros revelaram uma variação diária maior que a esperada.
    Os aparelhos registraram medições a cada quinze minutos.
    A equipe instalou sensores de temperatura na floresta em janeiro.


### Lendo os resultados

✅ **Onde funciona — o teste de embaralhamento passa de forma limpa.** Entre as **24** ordenações
possíveis do parágrafo, a original fica em **1º lugar**. Isso é um resultado forte: o log-prob
condicional é sensível à *ordem*, não só ao conteúdo — as 24 permutações têm exatamente as mesmas
palavras. O que separa a primeira da última é só o encadeamento, e o modelo o percebe. E o ranking
geral respeita a intuição: `coerente` (−1,97) > `embaralhado` (−2,33) > `sem relação` (−2,98).

❌ **Onde falha — e aqui os dois medidores erram, em direções opostas:**

1. **A coesão lexical inverte completamente o ranking.** O parágrafo **coerente** tira **0,00** —
   nenhum par de frases vizinhas compartilha um substantivo. E não é defeito dele: é *virtude*. Ele
   escreve bem, então varia o vocabulário — `sensores` vira `aparelhos`, depois `medições`, depois
   `registros`; `equipe` vira `grupo`. A cadeia de referência está lá, mas é **semântica**, e a
   comparação de lemas não a enxerga. Enquanto isso, o parágrafo **vazio**, que repete "os
   sensores" quatro vezes sem dizer nada, tira **1,00** — nota máxima. Sobreposição de superfície
   mede repetição, e repetição não é coerência. (O *entity grid* de verdade ataca isso resolvendo
   correferência; a versão por lemas, não.)

2. **A coerência por LM prefere o parágrafo vazio ao bom.** `coeso mas vazio` marca **−1,93**,
   acima do `coerente` (−1,97). O motivo é estrutural: log-prob condicional mede
   **previsibilidade**, e um texto repetitivo é o mais previsível que existe. Depois de três frases
   começando com "Os sensores", a quarta é quase gratuita. O medidor recompensa justamente o vício
   que um revisor humano marcaria primeiro. É o mesmo mecanismo do viés de verbosidade em juízes
   LLM: o que é fácil de prever ganha nota, e facilidade não é qualidade.

**O que esta camada estabelece:** coerência é a dimensão mais difícil e a menos confiável do
notebook. O teste de embaralhamento funciona porque compara **o mesmo conteúdo** em ordens
diferentes — de novo o princípio do par mínimo. Fora dessa comparação controlada, os dois
medidores são facilmente enganados, e nenhum deles distingue "texto que progride" de "texto que
se repete".

---
# Parte III — O juiz que cobre todas as dimensões

## Dimensão 7 — LLM com rubrica

Todos os instrumentos anteriores são especialistas: cada um vê uma dimensão e é cego às outras.
Um LLM com uma **rubrica** é o único juiz que responde sobre todas de uma vez — e em linguagem
natural, o que o torna o único capaz de *explicar* o problema a quem escreveu.

O preço: a nota vem de outro modelo falível, com vieses documentados. Os três principais:

| viés | sintoma | mitigação |
|---|---|---|
| posição | prefere a primeira opção apresentada | julgar 2× com a ordem trocada |
| verbosidade | prefere a resposta mais longa | rubrica que premie concisão |
| autopreferência | prefere o próprio estilo | juiz de família diferente da do avaliado |

A célula testa o juiz nas frases do corpus e depois mede o **viés de posição** diretamente: a
mesma comparação, duas vezes, com A e B trocados. Um juiz que julga o conteúdo aponta a mesma
frase nas duas ordens; um juiz que segue a posição, não.

In [ ]:
def gerar(prompt, max_new_tokens=60):
    enc = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        saida = modelo.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    texto = tokenizer.decode(saida[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return texto.strip().split("\n")[0].strip()


RUBRICA = (
    "Você é um revisor de textos em português. Avalie a FRASE nas dimensões:\n"
    "gramática, estrutura e sentido.\n"
    "Responda no formato: Nota: <1 a 5> - <problema em uma frase, ou 'sem problemas'>\n"
    "1 = quebrada; 3 = compreensível com erros; 5 = correta e com sentido claro.\n\n"
)

alvos = ["correta", "conc. verbal", "anomalia sem.", "vazia"]

print("--- formato A: prompt cru, o modelo completa 'Nota:' ---")
for nome in alvos:
    r = gerar(f"{RUBRICA}Frase: {frases[nome]}\nAvaliação:\nNota:", max_new_tokens=45)
    print(f"  {nome:14} -> Nota:{r}")

print("\n--- formato B: template de chat oficial do modelo ---")
INSTRUCAO = ("Você é um revisor de textos em português. Avalie a frase quanto a gramática, "
             "estrutura e sentido. Responda em UMA linha, no formato exato:\n"
             "Nota: <1 a 5> - <o problema, ou 'sem problemas'>")
for nome in alvos:
    msgs = [{"role": "system", "content": INSTRUCAO},
            {"role": "user", "content": f"Frase: {frases[nome]}"}]
    p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    print(f"  {nome:14} -> {gerar(p, max_new_tokens=45)}")


def comparar(a, b):
    p = (f"Qual frase está mais bem escrita em português?\n"
         f"Frase A: {a}\nFrase B: {b}\n"
         "Responda apenas A ou B.\nResposta:")
    return gerar(p, max_new_tokens=3)


print()
print("correta como A ->", comparar(frases["correta"], frases["conc. verbal"]))
print("correta como B ->", comparar(frases["conc. verbal"], frases["correta"]))
print("\njuiz consistente aponta a MESMA frase nas duas ordens;")
print("se a escolha acompanhar a posição (sempre A, ou sempre B), é viés de posição.")

--- formato A: prompt cru, o modelo completa 'Nota:' ---


  correta        -> Nota:5


  conc. verbal   -> Nota:4


  anomalia sem.  -> Nota:4


  vazia          -> Nota:4

--- formato B: template de chat oficial do modelo ---


...

  conc. verbal   -> Sem problemas


  anomalia sem.  -> Sem problemas


  vazia          -> Sem problemas



correta como A -> B


correta como B -> B

juiz consistente aponta a MESMA frase nas duas ordens;
se a escolha acompanhar a posição (sempre A, ou sempre B), é viés de posição.


### Lendo os resultados

✅ **Onde o protocolo funciona:** é o único instrumento do notebook que, *em princípio*, responde
sobre todas as dimensões de uma vez e em linguagem natural. Para revisão assistida isso vale mais
que qualquer métrica: "o verbo não concorda com o sujeito" é acionável, "SLOR = 4,95" não é. Com
juízes grandes, a correlação com avaliadores humanos é a mais alta entre todos os métodos
automáticos de qualidade aberta — é por isso que a família domina a prática atual.

E o **formato A** guarda um sinal: `correta` recebeu 5, e as três defeituosas receberam 4. A ordem
está certa.

❌ **Onde falha — e aqui a demonstração foi contra a expectativa:**

1. **O juiz não discriminou.** Um erro de concordância, uma anomalia semântica e uma frase sem
   sentido receberam a **mesma nota 4**. Três defeitos de naturezas completamente diferentes,
   indistinguíveis na saída. O parser, na Dimensão 2, separou dois deles apontando o token exato.
2. **Nenhuma justificativa saiu**, embora a rubrica a exigisse no formato. Pedimos "Nota: N -
   problema" e recebemos "Nota: N". A promessa do ✅ — o diagnóstico legível — **não foi cumprida
   por este juiz**.
3. **Trocar o formato do prompt destrói o resultado.** Mesmo modelo, mesma rubrica, mesmo conteúdo:
   pelo template de chat oficial, o **formato B responde "sem problemas" para as quatro frases** —
   inclusive para *"Os pesquisadores analisou"* e para *"beberam os dados"*. Se a avaliação fosse
   uma medida, a forma de perguntar não mudaria a resposta. Aqui muda tudo, e não há critério
   interno para decidir qual dos dois formatos é o "certo".
4. **Viés de posição, limpo.** O juiz respondeu **"B" nas duas ordens**. Como as frases trocaram de
   lugar entre as duas rodadas, responder sempre "B" significa que ele escolheu **frases
   diferentes** — ou seja, seguiu a *posição*, não o conteúdo. (Um juiz consistente faria o
   inverso: mudaria a letra para manter a mesma frase.) De quebra, isso mostra que a rodada em que
   ele "acertou" foi sorte de posicionamento.

**A comparação que fecha o notebook:** para `conc. verbal`, o parser respondeu em milissegundos,
sempre igual, apontando o token e o traço violado. O juiz levou segundos, não localizou nada, deu
a mesma nota que deu à frase sem sentido e mudou de opinião quando reformatamos o prompt.
**Quando existe um detector determinístico para a dimensão, ele ganha do LLM** — mais barato, mais
exato e mais explicativo.

**A lição não é "juiz não funciona".** É que a qualidade do julgamento escala com o juiz, e um
modelo de 1,5B avaliando gramática portuguesa está muito abaixo do necessário. Com GPT-4 ou Claude
no lugar, os quatro problemas acima encolhem — mas os vieses de posição e verbosidade **não
desaparecem**, e a sensibilidade ao formato do prompt tampouco. Julgar 2× com ordem trocada não é
preciosismo: é o mínimo.

---
# Parte IV — Validação e integração

## Como saber se o avaliador funciona

Um amontoado de pontuações não é um avaliador. Falta a pergunta de validação: **estes detectores
acertam?**

O corpus de pares mínimos foi construído exatamente para responder isso. Cada frase tem um defeito
conhecido (o dicionário `defeito`), então dá para montar a **matriz detector × defeito** e ler
duas coisas que nenhuma nota isolada mostra:

- **cobertura** — cada defeito é pego por alguém?
- **especificidade** — o detector dispara **só** no defeito dele, ou faz barulho em tudo?

A segunda coluna é a que separa um detector útil de um alarme inútil. Um detector que acusa todas
as frases tem cobertura perfeita e valor zero.

É a metodologia do **CoLA** e do **BLiMP**, os benchmarks de aceitabilidade: pares mínimos, e a
métrica é a proporção de pares ordenados corretamente. Para validação contra humanos, os
instrumentos são a **correlação de Spearman** (a ordem bate?) e o **Kappa de Cohen** entre
anotadores — se as pessoas não concordam entre si, nenhuma métrica automática vai correlacionar
bem, e o teto do que se pode exigir do detector é o acordo humano.

Os detectores desta matriz usam, em cada dimensão, a técnica que **venceu no estudo**: léxico
(ortografia), parser (gramática e estrutura) e **verbo mascarado com margem relativa** (semântica) —
esta última no lugar da surpresa com limiar fixo da Dimensão 5.

In [ ]:
# SEMANTICA da pipeline = verbo mascarado (vencedor do estudo), por MARGEM relativa
MARGEM_SEMANTICA = 10.0   # nats — corte relativo a posicao, nao um limiar absoluto de surpresa

def margem_verbo(frase):
    doc = nlp(frase)
    raiz = [t for t in doc if t.dep_ == "ROOT" and t.pos_ == "VERB"]
    if not raiz:
        return None                          # sem verbo na raiz (fragmento): nao se aplica
    alvo = raiz[0].text
    enc = tok_b(frase.replace(alvo, tok_b.mask_token, 1), return_tensors="pt")
    pos = (enc["input_ids"][0] == tok_b.mask_token_id).nonzero()
    if len(pos) == 0:
        return None
    with torch.no_grad():
        lp = torch.log_softmax(mod_b(**enc).logits[0, int(pos[0])], dim=-1)
    ida = tok_b(alvo, add_special_tokens=False)["input_ids"]
    return float(lp.max()) - float(lp[ida[0]])   # melhor esperado - verbo escrito


def detectores(frase):
    doc = nlp(frase)
    _, tem_predicado, _ = analisar_estrutura(doc)
    orto = bool(erros_ortograficos(frase))
    m = margem_verbo(frase)
    # gate de OOV: palavra fora do lexico ja e' pega pela ortografia e desregula a estatistica
    sem = (m is not None) and (not orto) and (m > MARGEM_SEMANTICA)
    return {
        "ortografia": orto,
        "gramática":  bool(erros_concordancia(doc)),
        "estrutura":  not tem_predicado,
        "semântica":  sem,
    }


nomes_det = ["ortografia", "gramática", "estrutura", "semântica"]
print(f"{'frase':14} {'defeito real':13} " + " ".join(f"{d[:6]:>7}" for d in nomes_det) + "   veredito")
print("-" * 76)

acertos = alarmes_falsos = 0
for nome, f in frases.items():
    d = detectores(f)
    esperado = defeito[nome]
    marcas = " ".join(f"{('  X' if d[k] else '  .'):>7}" for k in nomes_det)
    if esperado is None:
        ok = not any(d.values())
        veredito = "ok" if ok else "FALSO ALARME"
        alarmes_falsos += not ok
    else:
        ok = d.get(esperado, False)
        veredito = "pego" if ok else "ESCAPOU"
        acertos += ok
    print(f"{nome:14} {str(esperado or '—'):13} {marcas}   {veredito}")

n_defeituosas = sum(1 for v in defeito.values() if v)
print(f"\ncobertura: {acertos}/{n_defeituosas} defeitos pegos pelo detector da dimensão certa")
print(f"falsos alarmes em frases corretas: {alarmes_falsos}")

frase          defeito real   ortogr  gramát  estrut  semânt   veredito
----------------------------------------------------------------------------
correta        —                   .       .       .       .   ok
ortografia     ortografia          X       .       .       .   pego
conc. nominal  gramática           .       X       .       .   pego


conc. verbal   gramática           .       X       .       .   pego
fragmento      estrutura           .       .       X       .   pego
anomalia sem.  semântica           .       .       .       X   pego
embaralhada    estrutura           .       .       .       .   ESCAPOU
vazia          semântica           .       X       .       .   ESCAPOU

cobertura: 5/7 defeitos pegos pelo detector da dimensão certa
falsos alarmes em frases corretas: 0


### Lendo a matriz

A matriz é o resumo honesto do notebook — **cobertura 5/7, zero falso alarme na frase correta** — e
o que ela mostra de mais útil são as duas linhas que escapam.

**1. Onde funciona: a diagonal.** As cinco variantes de par mínimo são pegas, cada uma pelo detector
da sua dimensão: ortografia, concordância (nominal e verbal → gramática), fragmento (estrutura) e
anomalia semântica (verbo mascarado). A frase `correta` fica inteiramente limpa — nenhum detector
dispara à toa.

**2. `embaralhada` e `vazia` escapam — pelo mesmo motivo.** As duas frases sem defeito *localizado*
atravessam os quatro detectores sem serem pegas na etapa-alvo:

- `embaralhada` ("Dados os analisaram pesquisadores...") tem todas as palavras existentes,
  concordância que por acaso não colide, e o parser lhe impõe uma árvore de raiz verbal; o
  verbo-raiz, isolado, é plausível.
- `vazia` ("Ideias verdes incolores dormem furiosamente", Chomsky 1957) é **gramaticalmente
  perfeita**, e o verbo "dormem" cabe na posição — a etapa semântica, que só julga o verbo, não vê
  problema. O sem-sentido está na combinação dos adjetivos, não no verbo.

**Todo detector desta matriz procura um defeito localizado; a anomalia global não é local em lugar
nenhum.** Quem pega as duas é o **SLOR** — `embaralhada` −0,54 e `vazia` −0,90, ambos negativos —,
medida de aceitabilidade que não entra na matriz por não ser binária nem localizável. É o preço de
uma etapa semântica focada no verbo: ela ganha robustez contra a anomalia de seleção (pega
"beberam", "cantou" e "ferveram" por margem relativa, sem limiar absoluto), mas cede o nonsense
gramatical, que precisa de um sinal global.

**3. O falso positivo de gênero persiste.** `vazia` acende também em `gramática` — o erro de gênero
inexistente entre "incolores" e "verdes", que o parser inventa numa sequência improvável. É um
disparo indevido (a etapa-alvo de `vazia` é a semântica) e o mesmo modo de falha do parser em
substantivos de dois gêneros que a Dimensão 2 documentou.

**A conclusão metodológica:** a única razão de conseguirmos afirmar "escapou" e "falso positivo" é
que o corpus tem **gabarito**. Sem saber o defeito de cada frase, as quatro colunas seriam só X e
ponto, e os buracos seriam invisíveis. Construir o conjunto de teste é mais trabalho, e vale mais,
do que escolher a métrica.

In [ ]:
def laudo(frase):
    doc = nlp(frase)
    linhas, bloqueado = [], False

    orto = erros_ortograficos(frase)
    mec = erros_mecanicos(frase)
    linhas.append(("ortografia", "ERRO" if orto else "ok",
                   ", ".join(f"{p} → {c}" for p, c in orto.items()) or "—"))
    linhas.append(("mecânica", "ERRO" if mec else "ok", "; ".join(mec) or "—"))
    bloqueado = bool(orto)

    conc = erros_concordancia(doc)
    linhas.append(("gramática", "ERRO" if conc else "ok", "; ".join(conc) or "—"))

    raiz, pred, prof = analisar_estrutura(doc)
    linhas.append(("estrutura", "ERRO" if not pred else "ok",
                   f"raiz '{raiz.text}' não é verbo (fragmento)" if not pred
                   else f"raiz '{raiz.text}', profundidade {prof}"))

    m = margem_verbo(frase)
    if bloqueado:
        linhas.append(("semântica", "n/d", "não avaliada: palavra fora do léxico"))
    elif m is None:
        linhas.append(("semântica", "n/d", "sem verbo na raiz"))
    else:
        linhas.append(("semântica", "SUSPEITA" if m > MARGEM_SEMANTICA else "ok",
                       f"margem do verbo-raiz = {m:.1f} nats"))

    linhas.append(("aceitabilidade", "n/d" if bloqueado else "ok",
                   "não avaliada: há palavra fora do léxico" if bloqueado
                   else f"SLOR = {slor(frase):.2f}"))

    print(f'"{frase}"')
    for dim, status, det in linhas:
        print(f"   {dim:15} {status:9} {det}")
    print()


for nome in ["correta", "ortografia", "conc. verbal", "fragmento", "anomalia sem."]:
    laudo(frases[nome])

"Os pesquisadores analisaram os dados coletados durante o experimento."
   ortografia      ok        —
   mecânica        ok        —
   gramática       ok        —
   estrutura       ok        raiz 'analisaram', profundidade 4
   semântica       ok        margem do verbo-raiz = 4.1 nats
   aceitabilidade  ok        SLOR = 5.82

"Os pesquisadores analizaram os dados coletados durante o experimento."
   ortografia      ERRO      analizaram → analisaram
   mecânica        ok        —
   gramática       ok        —
   estrutura       ok        raiz 'analizaram', profundidade 4
   semântica       n/d       não avaliada: palavra fora do léxico
   aceitabilidade  n/d       não avaliada: há palavra fora do léxico



"Os pesquisadores analisou os dados coletados durante o experimento."
   ortografia      ok        —
   mecânica        ok        —
   gramática       ERRO      verbal: sujeito 'pesquisadores'=Plur × verbo 'analisou'=Sing
   estrutura       ok        raiz 'analisou', profundidade 4
   semântica       ok        margem do verbo-raiz = 7.5 nats
   aceitabilidade  ok        SLOR = 4.95



"Os pesquisadores que analisaram os dados coletados durante o experimento."
   ortografia      ok        —
   mecânica        ok        —
   gramática       ok        —
   estrutura       ERRO      raiz 'pesquisadores' não é verbo (fragmento)
   semântica       n/d       sem verbo na raiz
   aceitabilidade  ok        SLOR = 4.93



"Os pesquisadores beberam os dados coletados durante o experimento."
   ortografia      ok        —
   mecânica        ok        —
   gramática       ok        —
   estrutura       ok        raiz 'beberam', profundidade 4
   semântica       SUSPEITA  margem do verbo-raiz = 15.0 nats
   aceitabilidade  ok        SLOR = 3.80



## Síntese

**O avaliador de frases é um pipeline de especialistas, não uma nota.** A ordem não é arbitrária —
ela vai do mais barato, exato e localizável para o mais caro, vago e global, e cada etapa é
pré-requisito da seguinte. Cada dimensão usa a técnica que **venceu no estudo**:

| ordem | dimensão | juiz | custo | veredito | localiza? |
|---|---|---|---|---|---|
| 1 | ortografia / mecânica | léxico + regra | ~0 | binário | palavra exata |
| 2 | gramática | parser | baixo | binário | token + traço |
| 3 | estrutura | árvore de dependências | baixo | binário | raiz |
| 4 | aceitabilidade | LM causal (SLOR) | médio | contínuo | não |
| 5 | semântica | verbo mascarado (BERTimbau) | médio | margem relativa | verbo-raiz |
| 6 | coerência | LM condicional / entity grid | médio | contínuo | frase |
| 7 | tudo | LLM com rubrica | alto | nota (texto só se o juiz for grande) | em linguagem natural |

**Guia de decisão:**

1. Erro de **forma** (grafia, concordância, fragmento)? → etapas 1–3. Determinísticas, explicáveis,
   instantâneas. Não use LLM aqui: custa mais e acerta menos.
2. A frase está bem formada mas soa **estranha**? → verbo mascarado na posição da raiz (margem
   relativa, não limiar absoluto) e SLOR entre pares mínimos. Anomalias sem endereço no verbo —
   nonsense gramatical, desordem global — não caem aqui; precisam do sinal global do SLOR.
3. O objeto é um **texto**, não uma frase? → coerência (etapa 6), sempre validada por teste de
   embaralhamento. Desconfie de coesão lexical: ela premia repetição.
4. Precisa de um **diagnóstico legível** para quem escreveu? → etapa 7, com rubrica e ordem
   trocada nas comparações — e com um juiz grande. O de 1,5B testado aqui não entregou
   justificativa nenhuma e deu a mesma nota a três defeitos diferentes.
5. Vai comparar sistemas ou calibrar limiares? → construa um conjunto com **gabarito**. Sem ele,
   nada acima é verificável.

**As três lições que atravessam o notebook:**

- **O par mínimo é o instrumento central.** Toda medida confiável aqui veio de comparar duas
  versões que diferem em uma coisa só: as frases do corpus, as 24 permutações do parágrafo, as duas
  ordens do juiz. Fora dessa comparação controlada, os números absolutos não significam quase nada
  — não existe "SLOR = 4,9 é bom".
- **Diagnóstico e percepção são trocas opostas.** O parser diz exatamente o que está errado e não
  faz ideia do que a frase significa; o modelo de linguagem percebe que algo está errado e não sabe
  dizer o quê. Um avaliador útil precisa dos dois, e é por isso que ele é um pipeline.
- **A ordem do pipeline não é estética, é técnica.** A Dimensão 4 mostrou o SLOR **premiando** um
  erro de grafia, porque palavra fora do vocabulário desregula o termo de frequência. Rodar o
  corretor antes não é organização: é condição de validade da etapa seguinte. Por isso o `laudo`
  se recusa a reportar SLOR — e a etapa semântica — quando há palavra desconhecida.
- **Todo detector daqui procura defeito *localizado* — e por isso a anomalia global escapa.**
  `embaralhada` e `vazia` atravessaram os quatro detectores da matriz sem disparar na etapa-alvo,
  porque não há um token culpado: o problema está na relação entre as palavras (ordem) ou no sentido
  do conjunto (nonsense). Foi preciso uma medida global e não localizável — o SLOR, **−0,54** e
  **−0,90** — para flagrá-las. Quando desenhar um avaliador, pergunte quais defeitos do seu domínio
  **não têm endereço**: esses precisam de um instrumento de outra natureza.

**Limites honestos deste notebook.** O corpus tem oito frases — serve para *demonstrar mecanismos*,
não para estimar desempenho; números reais exigem centenas de pares anotados. O juiz de 1,5B está
muito abaixo do que se usa a sério. Não há resolução de correferência, sem a qual o *entity grid*
fica pela metade. E ferramentas maduras que este notebook reimplementa de forma didática —
**LanguageTool** para regras de gramática do português, **Coh-Metrix-Port** (NILC/USP) para coesão
e coerência — são o ponto de partida certo para uso real.